# Advanced Prompt Engineering Techniques

<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width="400px" style="opacity:0.7">
</center>

In [1]:
%run supportvectors-common.ipynb


<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors AI Lab. <blockquote>This notebook is part of the AI Lab Book companion repository. 
Code is released under the MIT License and accompanying text under CC BY 4.0 - see the LICENSE files at the repository root.

 </blockquote>
 <hr/>
</div>



This document provides detailed explanations and examples of various advanced prompt engineering techniques used to improve the performance and reliability of large language models.

## Setup and Configuration

First, let's set up the necessary imports and API configuration using the Instructor framework. Calls go to `openai/gpt-oss-20b` on the SupportVectors cluster (WireGuard required; `OPENAI_BASE_URL` and a dummy `OPENAI_API_KEY` come from `.env`).

In [2]:
import os
from typing import Any, Iterable, List, Optional
from pydantic import BaseModel, Field
import instructor
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam, ChatCompletionUserMessageParam
from rich import print as rprint

# SupportVectors cluster (WireGuard required). .env sets
# OPENAI_BASE_URL=http://10.0.10.70:8000/v1 and a dummy OPENAI_API_KEY.
# from_openai() returns a typed Instructor client. patch() returns OpenAI, so the
# type checker still sees Completions.create, which has no response_model argument.
# JSON mode: gpt-oss-20b emits multiple tool calls, which Instructor rejects.
client = instructor.from_openai(
    OpenAI(
        base_url=os.environ["OPENAI_BASE_URL"],
        api_key=os.environ["OPENAI_API_KEY"],
    ),
    mode=instructor.Mode.JSON,
)

def get_completion(
    prompt: str,
    model: str = "openai/gpt-oss-20b",
    temperature: float = 0.7,
    response_model: Optional[type[BaseModel] | type[Iterable[Any]]] = None,
) -> Any:
    """Helper function to get a completion, parsed with Instructor when a model is given."""
    messages: list[ChatCompletionMessageParam] = [
        ChatCompletionUserMessageParam(role="user", content=prompt)
    ]
    if response_model is not None:
        return client.chat.completions.create(
            response_model=response_model,
            model=model,
            messages=messages,
            temperature=temperature,
        )
    response = client.chat.completions.create(
        response_model=None,
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return response.choices[0].message.content

## 1. Chain of Thought (CoT)

Chain of Thought prompting encourages the model to break down complex problems into intermediate reasoning steps before arriving at the final answer.

### Description
- Makes the model's reasoning process explicit
- Helps in solving complex problems that require multiple steps
- Improves accuracy by allowing the model to work through problems step by step

### Example
```
Question: A company has 3 departments with 25, 30, and 35 employees respectively. Each department needs to send 20% of its employees to a training program. The training program costs $500 per person. What is the total cost of the training program?

Let's solve this step by step:
1. First, let's understand what we need to find
2. Then, we'll calculate how many employees from each department need training
3. Next, we'll find the total number of employees in training
4. Finally, we'll calculate the total cost
```

### Implementation

In [3]:
class CoTResponse(BaseModel):
    steps: List[str] = Field(description="List of reasoning steps")
    final_answer: str = Field(description="The final answer after reasoning")

def chain_of_thought_example():
    prompt = """
    Question: A company has 3 departments with 25, 30, and 35 employees respectively. Each department needs to send 20% of its employees to a training program. The training program costs $500 per person. What is the total cost of the training program?
    
    Let's solve this step by step:
    """
    
    response = get_completion(
        prompt,
        temperature=0.3,
        response_model=CoTResponse
    )
    rprint("Steps:", response.steps)
    rprint("Final Answer:", response.final_answer)

# Run the example
chain_of_thought_example()


Steps:
[
    'Determine the number of employees in each department: 25, 30, and 35.',
    "Calculate 20% of each department's employees: 20% of 25 = 5, 20% of 30 = 6, 20% of 35 = 7.",
    'Sum the participants from all departments: 5 + 6 + 7 = 18 employees will attend the training.',
    'Multiply the total number of participants by the cost per person: 18 × $500 = $9,000.'
]

Final Answer: $9,000

## 2. Tree of Thought (ToT)

Tree of Thought extends Chain of Thought by exploring multiple reasoning paths simultaneously, allowing the model to consider different approaches to solving a problem.

### Description
- Enables parallel exploration of different solution paths
- Helps in finding optimal solutions by comparing multiple approaches
- Useful for problems with multiple valid solutions or approaches

### Example
```
Question: If a train travels at 60 mph for 2.5 hours, how far does it go?

Let's explore different solution paths. For each path:
1. Give it a descriptive name
2. List the steps you'll take
3. Show your calculations
4. State the result

Then, evaluate all paths and choose the best one, explaining why it's the best approach.
```

### Implementation

In [4]:
class SolutionPath(BaseModel):
    path_name: str = Field(description="Name of the solution path")
    steps: List[str] = Field(description="Steps in this solution path")
    result: float = Field(description="Result of this path")

class ToTResponse(BaseModel):
    paths: List[SolutionPath] = Field(description="Different solution paths explored")
    best_path: str = Field(description="The selected best path")
    final_answer: float = Field(description="The final answer")
    reasoning: str = Field(description="Reasoning for selecting the best path")

def tree_of_thought_example():
    prompt = """
    Question: If a train travels at 60 mph for 2.5 hours, how far does it go?

    Let's explore different solution paths. For each path:
    1. Give it a descriptive name
    2. List the steps you'll take
    3. Show your calculations
    4. State the result

    Then, evaluate all paths and choose the best one, explaining why it's the best approach.
    """
    
    try:
        response = get_completion(
            prompt,
            temperature=0.3,
            response_model=ToTResponse
        )
        rprint("Solution Paths:", response.paths)
        rprint("Best Path:", response.best_path)
        rprint("Final Answer:", response.final_answer)
        rprint("Reasoning:", response.reasoning)
    except Exception as e:
        rprint(f"Error in Tree of Thought example: {str(e)}")
        rprint("Please check the response model structure and prompt format.")

# Run the example
tree_of_thought_example()

Solution Paths:
[
    SolutionPath(
        path_name='Direct Multiplication',
        steps=[
            'Identify the given speed: 60 mph.',
            'Identify the given time: 2.5 hours.',
            'Multiply speed by time: 60 mph × 2.5 h.',
            'Compute the product: 60 × 2.5 = 150.'
        ],
        result=150.0
    ),
    SolutionPath(
        path_name='Unit Conversion',
        steps=[
            'Convert the speed to miles per minute: 60 miles ÷ 60 minutes = 1 mile/min.',
            'Convert the time to minutes: 2.5 hours × 60 minutes/hour = 150 minutes.',
            'Multiply miles per minute by minutes: 1 mile/min × 150 min.',
            'Compute the product: 1 × 150 = 150.'
        ],
        result=150.0
    ),
    SolutionPath(
        path_name='Average Speed Concept',
        steps=[
            'Recall the distance formula: distance = speed × time.',
            'Insert the given values: distance = 60 mph × 2.5 h.',
            'Perform the multiplication: 60 × 2.5 = 150.'
        ],
        result=150.0
    )
]

Best Path: Direct Multiplication

Final Answer: 150.0

Reasoning: The Direct Multiplication path uses the simplest and most direct application of the distance formula 
(distance = speed × time). It requires only one multiplication step and no unit conversions, making it the most 
straightforward and least error‑prone approach among the explored paths.

## 3. Few-Shot Prompting

Few-shot prompting provides the model with a few examples of the desired input-output pairs before asking it to perform the task.

### Description
- Helps the model understand the expected format and style
- Improves performance on specific tasks
- Reduces the need for explicit instructions

### Example
```
Task: Analyze the sentiment and key topics in customer feedback.

Example 1:
Input: "The new mobile app interface is intuitive and user-friendly. However, the login process takes too long and sometimes fails. The customer support team was very helpful in resolving my issues."
Output: {
    "sentiment": "mixed",
    "key_topics": ["mobile app", "login process", "customer support"],
    "positive_aspects": ["interface", "customer support"],
    "negative_aspects": ["login process"],
    "suggestions": ["optimize login process", "maintain current interface design"]
}

Example 2:
Input: "The product quality is excellent and the delivery was fast. I'm impressed with the packaging and the attention to detail. Will definitely order again!"
Output: {
    "sentiment": "positive",
    "key_topics": ["product quality", "delivery", "packaging"],
    "positive_aspects": ["product quality", "delivery speed", "packaging"],
    "negative_aspects": [],
    "suggestions": ["maintain current quality standards"]
}

Example 3:
Input: "The website is constantly crashing and the search function doesn't work properly. The prices are reasonable but the overall experience is frustrating. I've contacted support multiple times with no response."
Output: {
    "sentiment": "negative",
    "key_topics": ["website stability", "search function", "pricing", "customer support"],
    "positive_aspects": ["pricing"],
    "negative_aspects": ["website stability", "search function", "customer support"],
    "suggestions": ["fix website stability issues", "improve search functionality", "enhance customer support response time"]
}

Now, analyze this new feedback:
Input: "The new dashboard layout is confusing and not intuitive. The data visualization features are great, but they're hard to find. The export functionality works well, but the file formats are limited. Overall, it needs improvement in user experience."
Output:
```

### Implementation

In [5]:
class FeedbackAnalysis(BaseModel):
    sentiment: str = Field(description="Overall sentiment of the feedback (positive/negative/mixed)")
    key_topics: List[str] = Field(description="Main topics discussed in the feedback")
    positive_aspects: List[str] = Field(description="Positive aspects mentioned")
    negative_aspects: List[str] = Field(description="Negative aspects mentioned")
    suggestions: List[str] = Field(description="Suggested improvements based on the feedback")

def few_shot_example():
    prompt = """
    Task: Analyze the sentiment and key topics in customer feedback.

    Example 1:
    Input: "The new mobile app interface is intuitive and user-friendly. However, the login process takes too long and sometimes fails. The customer support team was very helpful in resolving my issues."
    Output: {
        "sentiment": "mixed",
        "key_topics": ["mobile app", "login process", "customer support"],
        "positive_aspects": ["interface", "customer support"],
        "negative_aspects": ["login process"],
        "suggestions": ["optimize login process", "maintain current interface design"]
    }

    Example 2:
    Input: "The product quality is excellent and the delivery was fast. I'm impressed with the packaging and the attention to detail. Will definitely order again!"
    Output: {
        "sentiment": "positive",
        "key_topics": ["product quality", "delivery", "packaging"],
        "positive_aspects": ["product quality", "delivery speed", "packaging"],
        "negative_aspects": [],
        "suggestions": ["maintain current quality standards"]
    }

    Example 3:
    Input: "The website is constantly crashing and the search function doesn't work properly. The prices are reasonable but the overall experience is frustrating. I've contacted support multiple times with no response."
    Output: {
        "sentiment": "negative",
        "key_topics": ["website stability", "search function", "pricing", "customer support"],
        "positive_aspects": ["pricing"],
        "negative_aspects": ["website stability", "search function", "customer support"],
        "suggestions": ["fix website stability issues", "improve search functionality", "enhance customer support response time"]
    }

    Now, analyze this new feedback:
    Input: "The new dashboard layout is confusing and not intuitive. The data visualization features are great, but they're hard to find. The export functionality works well, but the file formats are limited. Overall, it needs improvement in user experience."
    Output:
    """
    
    response = get_completion(
        prompt,
        temperature=0.3,
        response_model=FeedbackAnalysis
    )
    rprint("Sentiment:", response.sentiment)
    rprint("Key Topics:", response.key_topics)
    rprint("Positive Aspects:", response.positive_aspects)
    rprint("Negative Aspects:", response.negative_aspects)
    rprint("Suggestions:", response.suggestions)

# Run the example
few_shot_example()

Sentiment: mixed

Key Topics:
['dashboard layout', 'data visualization', 'export functionality', 'file formats', 'user experience']

Positive Aspects:
['data visualization features', 'export functionality']

Negative Aspects:
['dashboard layout', 'hard to find features', 'limited file formats', 'overall user experience']

Suggestions:
[
    'improve dashboard layout for better intuitiveness',
    'enhance discoverability of data visualization features',
    'add more export file formats',
    'overall improve user experience'
]

## 4. ReAct (Reasoning and Acting)

ReAct combines reasoning and action-taking capabilities, allowing the model to interact with external tools or knowledge bases while solving problems.

### Description
- Integrates reasoning with action-taking
- Enables dynamic problem-solving
- Useful for tasks requiring external information or tools

### Example
```
Question: A user reports that their application is running slowly. The application is a web service that processes customer data. The user mentions that the slowdown started after a recent deployment. How would you diagnose and fix this issue?

Let's solve this step by step:

Thought: First, I need to understand the current state of the system
Action: Check system metrics and logs
Observation: CPU usage is at 90%, memory usage is at 80%, and there are many database connection timeouts

Thought: The high CPU and memory usage suggests a resource bottleneck
Action: Analyze recent code changes in the deployment
Observation: The new code includes a change in the database query pattern that removed indexing

Thought: The missing index is likely causing the slowdown
Action: Review the database schema and query patterns
Observation: The query is performing full table scans instead of using indexes

Thought: I can now propose a solution
Answer: The performance issue is caused by a missing database index after the recent deployment. The solution is to:
1. Add the missing index to the database
2. Monitor the system after applying the fix
3. Consider adding query optimization to prevent similar issues in the future
```

### Implementation

In [6]:
from typing import Literal

# Labels that simulate_action knows how to answer. The model must pick one
# so the tool result is a real observation instead of a generic success message.
ActionDescription = Literal[
    "check system metrics",
    "analyze code changes",
    "review database schema",
    "check logs",
]

class ReActStep(BaseModel):
    thought: str = Field(description="The reasoning step")
    action: str = Field(description="The specific command or query to run for the chosen action")
    action_description: ActionDescription = Field(
        description=(
            "Which tool this action is. Must be exactly one of: "
            "check system metrics, analyze code changes, review database schema, check logs"
        )
    )
    observation: str = Field(description="The observation from the action")

class ReActResponse(BaseModel):
    steps: List[ReActStep] = Field(description="List of ReAct steps")
    final_answer: str = Field(description="The final answer")

class FinalAnswer(BaseModel):
    final_answer: str = Field(description="The diagnosis and the fix, based on the recorded steps")

def react_example():
    # Initial prompt
    prompt = """
    Question: A user reports that their application is running slowly. The application is a web service that processes customer data. The user mentions that the slowdown started after a recent deployment. How would you diagnose and fix this issue?
    
    Let's solve this step by step. For each step:
    1. Think about what information we need or what action to take
    2. Take an action using one of the available tools
    3. Observe the results
    4. Use the observations to inform the next step

    Available tools. Set action_description to exactly one of these:
    - check system metrics
    - analyze code changes
    - review database schema
    - check logs

    Return one step only, with these fields:
    - thought: your reasoning
    - action: the specific command or query for the chosen tool
    - action_description: exactly one of the tool labels above
    - observation: leave a short placeholder; the tool result replaces it
    """
    
    # Steps already taken. Observations come from the tool, not from a second model call.
    steps: List[ReActStep] = []
    conversation_history = []
    max_steps = 5  # Prevent infinite loops
    current_step = 0
    
    while current_step < max_steps:
        # Get next step from LLM
        response = get_completion(
            prompt + "\n\nCurrent conversation history:\n" + "\n".join(conversation_history),
            temperature=0.3,
            response_model=ReActStep
        )
        
        # Simulate action and get observation.
        # Match on action_description, which is constrained to the tool labels
        # simulate_action understands. The free-form action string is the command.
        action_result = simulate_action(response.action_description)
        step = ReActStep(
            thought=response.thought,
            action=response.action,
            action_description=response.action_description,
            observation=action_result,
        )
        steps.append(step)
        conversation_history.append(f"Thought: {step.thought}")
        conversation_history.append(f"Action ({step.action_description}): {step.action}")
        conversation_history.append(f"Observation: {step.observation}")
        
        # Check if we've reached a conclusion
        if "final answer" in response.thought.lower() or "solution" in response.thought.lower():
            break
            
        current_step += 1
    
    # Ask only for the answer. Rebuilding `steps` here makes the model collapse
    # each trace into a single `step` string, which fails ReActStep validation.
    final = get_completion(
        prompt + "\n\nBased on the following steps, provide a final answer:\n" + "\n".join(conversation_history),
        temperature=0.3,
        response_model=FinalAnswer
    )
    result = ReActResponse(steps=steps, final_answer=final.final_answer)
    
    rprint("Steps:", result.steps)
    rprint("Final Answer:", result.final_answer)

def simulate_action(action_description: str) -> str:
    """Simulate the result of an action. In a real implementation, this would call actual tools.

    Match on action_description, not the free-form command. The model is asked to
    set that field to one of the labels below, so each step gets a specific observation.
    """
    action_description = action_description.lower()
    if "check system metrics" in action_description:
        return "CPU usage: 90%, Memory usage: 80%, Database connections: 100/100"
    elif "analyze code changes" in action_description:
        return "Recent deployment removed database indexing on customer_data table"
    elif "review database schema" in action_description:
        return "Query is performing full table scans instead of using indexes"
    elif "check logs" in action_description:
        return "Multiple database connection timeouts detected"
    else:
        return "Action completed successfully"

# Run the example
react_example()

Steps:
[
    ReActStep(
        thought='The slowdown likely correlates with resource usage spikes after deployment. First, gather current 
system metrics to identify any abnormal CPU, memory, or I/O patterns that could explain the latency.',
        action='fetch_system_metrics',
        action_description='check system metrics',
        observation='CPU usage: 90%, Memory usage: 80%, Database connections: 100/100'
    ),
    ReActStep(
        thought='The high CPU and memory usage suggest the new deployment introduced a performance regression. To 
pinpoint the cause, I need to review the recent code changes that were deployed.',
        action='analyze_code_changes',
        action_description='analyze code changes',
        observation='Recent deployment removed database indexing on customer_data table'
    ),
    ReActStep(
        thought='The recent deployment removed an index on the customer_data table, which likely caused the 
database to perform full table scans and increased query latency. To confirm this and identify the specific slow 
queries, I will check the application and database logs for any long-running queries or errors.',
        action='check_logs',
        action_description='check logs',
        observation='Multiple database connection timeouts detected'
    ),
    ReActStep(
        thought='The logs show multiple database connection timeouts, likely due to slow queries after the index 
was removed. To confirm the missing index and plan a fix, I need to review the current database schema for the 
customer_data table.',
        action='show_indexes customer_data',
        action_description='review database schema',
        observation='Query is performing full table scans instead of using indexes'
    ),
    ReActStep(
        thought='The logs indicate timeouts, but to confirm which queries are slow and which fields lack indexing, 
I should inspect the slow query log. This will help identify the exact query patterns that are causing full table 
scans and guide the index creation.',
        action='check_logs slow_query_log',
        action_description='check logs',
        observation='Multiple database connection timeouts detected'
    )
]

Final Answer: The application slowdown is caused by a missing index on the `customer_data` table that was removed 
during the recent deployment. Without the index, queries perform full table scans, driving up CPU, memory usage, 
and causing database connection timeouts. Re‑create the appropriate index (e.g., `CREATE INDEX 
idx_customer_data_on_columnX ON customer_data(columnX);` – replace `columnX` with the actual column used in the 
slow queries), redeploy the application, and verify that system metrics return to normal and that query latency 
drops to acceptable levels.

## 5. Reflexion

Reflexion prompts the model to reflect on its own outputs and reasoning process, enabling self-improvement and error correction.

### Description
- Encourages self-evaluation and improvement
- Helps identify and correct mistakes
- Useful for complex, multi-step problems

### Example
```
Question: Solve the equation: 2x + 5 = 15

Initial Solution:
2x + 5 = 15
2x = 10
x = 5

Reflection:
Let me verify my solution:
1. If x = 5, then 2(5) + 5 = 15
2. 10 + 5 = 15
3. 15 = 15 ✓

The solution is correct because it satisfies the original equation.
```

### Implementation

In [7]:
class InitialSolution(BaseModel):
    solution: str = Field(description="The initial solution to the problem")
    steps: List[str] = Field(description="Steps taken to reach the solution")

class ReflexionResponse(BaseModel):
    initial_solution: str = Field(description="The initial solution")
    verification_steps: List[str] = Field(description="Steps to verify the solution")
    is_correct: bool = Field(description="Whether the solution is correct")
    explanation: str = Field(description="Explanation of the verification")
    corrected_solution: Optional[str] = Field(description="Corrected solution if the initial one was wrong")

def reflexion_example():
    # Step 1: Get initial solution
    solution_prompt = """
    Question: Solve the equation: 2x + 5 = 15
    
    Please solve this step by step, showing your work.
    """
    
    initial_solution = get_completion(
        solution_prompt,
        temperature=0.3,
        response_model=InitialSolution
    )
    
    # Step 2: Ask for reflection and verification
    reflection_prompt = f"""
    You previously solved the equation: 2x + 5 = 15
    Your solution was: {initial_solution.solution}
    Your steps were: {initial_solution.steps}
    
    Please reflect on your solution and provide a verification steps.
    """
    
    reflection = get_completion(
        reflection_prompt,
        temperature=0.3,
        response_model=ReflexionResponse
    )
    
    rprint("Initial Solution:", initial_solution.solution)
    rprint("Solution Steps:", initial_solution.steps)
    rprint("\nVerification Steps:", reflection.verification_steps)
    rprint("Is Correct:", reflection.is_correct)
    rprint("Explanation:", reflection.explanation)
    if reflection.corrected_solution:
        rprint("Corrected Solution:", reflection.corrected_solution)

# Run the example
reflexion_example()

Initial Solution: x = 5

Solution Steps:
['Subtract 5 from both sides: 2x + 5 - 5 = 15 - 5 → 2x = 10', 'Divide both sides by 2: 2x ÷ 2 = 10 ÷ 2 → x = 5']

Verification Steps:
[
    'Subtract 5 from both sides: 2x + 5 - 5 = 15 - 5, which simplifies to 2x = 10.',
    'Divide both sides by 2: 2x ÷ 2 = 10 ÷ 2, which simplifies to x = 5.',
    'Substitute x = 5 back into the original equation: 2(5) + 5 = 10 + 5 = 15, which matches the right‑hand side.'
]

Is Correct: True

Explanation: The steps correctly isolate x and the resulting value satisfies the original equation, confirming the 
solution is correct.

## 6. Prompt Chaining

Prompt Chaining breaks down complex tasks into a series of simpler prompts, where the output of one prompt becomes the input for the next.

### Description
- Decomposes complex tasks into manageable steps
- Maintains context between related prompts
- Useful for multi-stage processes

### Example
```
Step 1: Summarize the following text
Input: [Long article about climate change]
Output: [Concise summary]

Step 2: Extract key points from the summary
Input: [Summary from Step 1]
Output: [List of key points]

Step 3: Generate questions based on key points
Input: [Key points from Step 2]
Output: [Relevant questions]

Final Output: A structured set of questions based on the original article's key points
```

### Implementation

In [8]:
class SummaryResponse(BaseModel):
    summary: str = Field(description="The summarized text")

class KeyPointsResponse(BaseModel):
    key_points: List[str] = Field(description="List of key points")

class QuestionsResponse(BaseModel):
    questions: List[str] = Field(description="List of generated questions")

def prompt_chaining_example():
    # Step 1: Summarize text
    text = "Climate change is a pressing global issue. Rising temperatures, extreme weather events, and sea level rise are affecting ecosystems worldwide. Scientists warn that immediate action is needed to reduce greenhouse gas emissions and transition to renewable energy sources."
    
    summary_prompt = f"Summarize the following text in one sentence:\n{text}"
    summary = get_completion(
        summary_prompt,
        temperature=0.3,
        response_model=SummaryResponse
    ).summary
    
    # Step 2: Extract key points
    key_points_prompt = f"Extract 3 key points from this summary:\n{summary}"
    key_points = get_completion(
        key_points_prompt,
        temperature=0.3,
        response_model=KeyPointsResponse
    ).key_points
    
    # Step 3: Generate questions
    questions_prompt = f"Generate 2 questions based on these key points:\n{key_points}"
    questions = get_completion(
        questions_prompt,
        temperature=0.7,
        response_model=QuestionsResponse
    ).questions
    
    rprint("Summary:", summary)
    rprint("\nKey Points:", key_points)
    rprint("\nQuestions:", questions)

# Run the example
prompt_chaining_example()

Summary: Climate change, driven by rising temperatures, extreme weather, and sea level rise, urgently requires 
reducing greenhouse gas emissions and shifting to renewable energy to protect global ecosystems.

Key Points:
[
    'Climate change is driven by rising temperatures, extreme weather, and sea level rise.',
    'There is an urgent need to reduce greenhouse gas emissions.',
    'Shifting to renewable energy is essential to protect global ecosystems.'
]

Questions:
[
    'How does the increase in global temperatures contribute to extreme weather events and rising sea levels?',
    'Why is transitioning to renewable energy crucial for reducing greenhouse gas emissions and safeguarding global
ecosystems?'
]

## 7. Self-Consistency

Self-Consistency involves generating multiple reasoning paths and selecting the most consistent answer among them.

### Description
- Generates multiple diverse reasoning paths
- Reduces errors by selecting the most consistent answer
- Improves reliability of complex problem-solving

### Example
```
Question: If a train travels at 60 mph for 2.5 hours, how far does it go?

Solution Path 1:
1. Distance = Speed × Time
2. Distance = 60 mph × 2.5 hours
3. Distance = 150 miles

Solution Path 2:
1. In 1 hour, train travels 60 miles
2. In 2 hours, train travels 120 miles
3. In 0.5 hours, train travels 30 miles
4. Total distance = 120 + 30 = 150 miles

Both paths lead to 150 miles, confirming the answer is correct.
```

### Implementation

In [9]:
class SolutionPath(BaseModel):
    path_name: str = Field(description="Name of the solution path")
    steps: List[str] = Field(description="Steps in the solution path")
    answer: float = Field(description="The final answer")

class SelfConsistencyResponse(BaseModel):
    paths: List[SolutionPath] = Field(description="Different solution paths")
    final_answer: float = Field(description="The consistent final answer")
    is_consistent: bool = Field(description="Whether all paths lead to the same answer")
    reasoning: str = Field(description="Explanation of why the paths are consistent or not")

def self_consistency_example():
    # Step 1: Generate multiple solution paths
    paths_prompt = """
    Question: If a train travels at 60 mph for 2.5 hours, how far does it go?

    Please solve this problem in 3 different ways. For each solution path:
    1. Give it a descriptive name
    2. List the steps you'll take
    3. Show your calculations
    4. State the result

    Make sure to use different approaches for each path.
    """
    
    # Get multiple solution paths
    paths_response = get_completion(
        paths_prompt,
        temperature=0.7,  # Higher temperature for more diverse solutions
        response_model=SelfConsistencyResponse
    )
    
    # Step 2: Analyze consistency
    consistency_prompt = f"""
    You generated the following solution paths for the problem "If a train travels at 60 mph for 2.5 hours, how far does it go?":

    {paths_response.paths}

    Please analyze these paths:
    1. Compare the answers from each path
    2. Determine if they are consistent
    3. If they are consistent, explain why this gives us confidence in the answer
    4. If they are not consistent, identify which path is correct and why
    """
    
    consistency_analysis = get_completion(
        consistency_prompt,
        temperature=0.3,  # Lower temperature for more focused analysis
        response_model=SelfConsistencyResponse
    )
    
    rprint("Solution Paths:")
    for path in paths_response.paths:
        rprint(f"\nPath: {path.path_name}")
        rprint("Steps:", path.steps)
        rprint("Answer:", path.answer)
    
    rprint("\nConsistency Analysis:")
    rprint("Final Answer:", consistency_analysis.final_answer)
    rprint("Is Consistent:", consistency_analysis.is_consistent)
    rprint("Reasoning:", consistency_analysis.reasoning)

# Run the example
self_consistency_example()

Solution Paths:

Path: Direct Multiplication

Steps:
[
    'Identify the speed: 60 miles per hour.',
    'Identify the travel time: 2.5 hours.',
    'Multiply speed by time: 60 miles/hour × 2.5 hours.',
    'Compute the product: 60 × 2.5 = 150 miles.'
]

Answer: 150.0

Path: Unit Conversion to Minutes

...

Answer: 150.0

Consistency Analysis:

Final Answer: 150.0

Is Consistent: True

Reasoning: All three solution paths yield the same numerical answer of 150 miles. The first path uses 
straightforward multiplication, the second verifies the result by converting to minutes and confirming the same 
product, and the third applies the standard rate‑time‑distance formula. The agreement across independent methods 
confirms the correctness of the calculation and gives confidence that the answer is reliable.

## 8. Least-to-Most Prompting

Least-to-Most prompting breaks down complex problems into simpler sub-problems that are solved sequentially.

### Description
- Decomposes complex problems into simpler steps
- Solves each sub-problem independently
- Builds up to the final solution

### Example
```
Question: Calculate the total cost of 3 shirts at $25 each with a 20% discount and 8% tax.

Step 1: Calculate base cost
3 shirts × $25 = $75

Step 2: Calculate discount
20% of $75 = $15

Step 3: Calculate tax
8% of $60 = $4.80

Step 4: Calculate final cost
$60 + $4.80 = $64.80
```

### Implementation

In [10]:
class CalculationStep(BaseModel):
    step_number: int = Field(description="Step number")
    description: str = Field(description="Description of the step")
    calculation: str = Field(description="The calculation")
    result: float = Field(description="Result of the calculation")

class LeastToMostResponse(BaseModel):
    steps: List[CalculationStep] = Field(description="List of calculation steps")
    final_result: float = Field(description="The final result")

def least_to_most_example():
    # Step 1: Break down the problem
    breakdown_prompt = """
    Question: Calculate the total cost of 3 shirts at $25 each with a 20% discount and 8% tax.

    Please break this problem down into the smallest possible steps that need to be solved in sequence.
    For each step, explain why it needs to be solved in that order.
    """
    
    breakdown = get_completion(
        breakdown_prompt,
        temperature=0.3,
        response_model=List[str]
    )
    
    # Step 2: Solve each step sequentially
    current_result = None
    steps = []
    
    for i, step_description in enumerate(breakdown, 1):
        step_prompt = f"""
        We are solving: Calculate the total cost of 3 shirts at $25 each with a 20% discount and 8% tax.
        
        Current step ({i}): {step_description}
        
        Previous steps results: {current_result if current_result is not None else 'No previous results'}
        
        Please solve this step, showing your calculation and result.
        """
        
        step_result = get_completion(
            step_prompt,
            temperature=0.3,
            response_model=CalculationStep
        )
        
        steps.append(step_result)
        current_result = step_result.result
    
    # Create final response
    response = LeastToMostResponse(
        steps=steps,
        final_result=current_result
    )
    
    rprint("Problem Breakdown:")
    for i, step in enumerate(breakdown, 1):
        rprint(f"{i}. {step}")
    
    rprint("\nSolution Steps:")
    for step in response.steps:
        rprint(f"\nStep {step.step_number}: {step.description}")
        rprint(f"Calculation: {step.calculation}")
        rprint(f"Result: {step.result}")
    
    rprint(f"\nFinal Result: ${response.final_result:.2f}")

# Run the example
least_to_most_example()

Problem Breakdown:

1. 1. Compute the initial subtotal by multiplying the quantity of shirts (3) by the unit price ($25). This gives 
the base amount before any discounts or taxes, which is necessary to know the starting point for further 
calculations.

2. 2. Calculate the discount amount by applying the 20% discount rate to the subtotal. The discount must be 
computed before tax because the tax is typically applied to the discounted price, not the original price.

3. 3. Subtract the discount amount from the subtotal to obtain the discounted subtotal. This step yields the amount
on which the tax will be calculated, ensuring the correct taxable base.

4. 4. Compute the tax by applying the 8% tax rate to the discounted subtotal. Tax is calculated after discounts to 
reflect the actual amount the customer pays for the goods.

...

Result: 4.8

Step 5: Add the tax amount to the discounted subtotal to arrive at the final total cost. This final step combines 
the discounted price and the tax to give the amount the customer must pay.

Calculation: 60 + 4.8 = 64.8

Result: 64.8

Final Result: $64.80

## 9. Zero-Shot Prompting

Zero-shot prompting asks the model to perform a task without any examples, relying on its pre-trained knowledge.

### Description
- No examples provided
- Relies on model's pre-trained knowledge
- Useful for straightforward tasks

### Example
```
Question: Translate this sentence to French: "Hello, how are you?"
Answer: "Bonjour, comment allez-vous?"
```

### Implementation


In [11]:

class TranslationResponse(BaseModel):
    original_text: str = Field(description="The original text")
    translated_text: str = Field(description="The translated text")
    language: str = Field(description="The target language")

def zero_shot_example():
    prompt = "Translate this sentence to French: 'Hello, how are you?'"
    
    response = get_completion(
        prompt,
        temperature=0.3,
        response_model=TranslationResponse
    )
    rprint("Original:", response.original_text)
    rprint("Translated:", response.translated_text)
    rprint("Language:", response.language)

# Run the example
zero_shot_example()

Original: Hello, how are you?

Translated: Bonjour, comment ça va ?

Language: French

## 10. Role Prompting

Role prompting assigns a specific role or persona to the model to guide its responses.

### Description
- Provides context through role assignment
- Influences response style and perspective
- Useful for specialized knowledge domains

### Example
```
You are an experienced chef with 20 years of experience in French cuisine. 
Please explain how to make a perfect béarnaise sauce.

[Model responds with detailed, expert-level instructions]
```

### Implementation


In [12]:
class RecipeResponse(BaseModel):
    title: str = Field(description="Name of the recipe")
    ingredients: List[str] = Field(description="List of ingredients")
    steps: List[str] = Field(description="Cooking steps")
    tips: List[str] = Field(description="Expert tips")

In [13]:
prompt_good = """
You are an experienced chef with 20 years of experience in French cuisine. 
    Please explain how to make a perfect béarnaise sauce.
"""

prompt_bad = """
You are a musician with 20 years of experience in music production. 
    Please explain how to make a perfect béarnaise sauce.
"""


In [14]:


def role_prompting_example(prompt:str):
    
    response = get_completion(
        prompt,
        temperature=0.1,
        response_model=RecipeResponse
    )
    rprint("Title:", response.title)
    rprint("Ingredients:", response.ingredients)
    rprint("Steps:", response.steps)
    rprint("Tips:", response.tips)


In [15]:

role_prompting_example(prompt_good)

Title: Béarnaise Sauce

Ingredients:
[
    '1/4 cup white wine vinegar',
    '1/4 cup dry white wine',
    '2 tablespoons chopped shallots',
    '2 tablespoons chopped fresh tarragon (plus extra for garnish)',
    '1 tablespoon chopped fresh chervil',
    '1 tablespoon chopped fresh parsley',
    '1 teaspoon fresh lemon juice',
    '1/2 teaspoon salt',
    '1/4 teaspoon freshly ground black pepper',
    '3 large egg yolks',
    '1/2 cup unsalted butter, melted and hot',
    '1 tablespoon cold water (optional, for adjusting consistency)'
]

Steps:
[
    'Combine the vinegar, white wine, shallots, half of the tarragon, chervil, parsley, lemon juice, salt, and 
pepper in a small saucepan.',
    'Simmer over medium heat until the liquid has reduced to about 2 tablespoons. Strain the mixture through a 
fine-mesh sieve into a heat‑proof bowl, discarding the solids. Let the reduction cool slightly.',
    'Place the bowl over a pot of gently simmering water (double boiler) and whisk in the egg yolks until the 
mixture is thick and pale.',
    'Slowly drizzle the hot melted butter into the yolk mixture while whisking continuously. Keep the whisk moving 
to prevent the eggs from scrambling.',
    'If the sauce becomes too thick, whisk in a tablespoon of cold water to reach a silky, pourable consistency.',
    'Remove the bowl from the heat and stir in the remaining fresh tarragon.',
    'Taste and adjust seasoning with additional salt, pepper, or lemon juice if needed.',
    'Serve immediately over steak, fish, or vegetables, or keep warm over a very low heat until ready to serve.'
]

Tips:
[
    'Use a double boiler to avoid overheating the eggs; high heat can cause curdling.',
    'Keep the reduction at a gentle simmer; a rapid boil will make the sauce bitter.',
    'If the sauce separates, whisk in a teaspoon of cold water or a small amount of warm stock to bring it back 
together.',
    'Fresh herbs are essential; dried herbs lose their bright flavor and can make the sauce taste flat.',
    'For a smoother sauce, strain the finished béarnaise through a fine sieve before serving.',
    'If you need to reheat the sauce, do so gently over a double boiler while whisking constantly to prevent it 
from breaking.'
]

In [17]:
role_prompting_example(prompt_bad)

Title: Béarnaise Sauce

Ingredients:
[
    '1/4 cup white wine vinegar',
    '1 tablespoon white wine',
    '1 tablespoon chopped shallots',
    '1 tablespoon chopped fresh tarragon (plus extra for garnish)',
    '1/2 teaspoon salt',
    '1/4 teaspoon freshly ground black pepper',
    '2 large egg yolks',
    '1/2 cup unsalted butter, melted and kept warm',
    '1 teaspoon water (optional, to adjust consistency)'
]

Steps:
[
    'In a small saucepan, combine the white wine vinegar, white wine, chopped shallots, chopped tarragon, salt, and
pepper. Bring to a gentle simmer over medium heat and let it reduce until the liquid is about 2 tablespoons.',
    'Strain the reduction through a fine-mesh sieve into a heat‑proof bowl, discarding the solids. Let the liquid 
cool slightly (it should still be warm but not boiling).',
    'Place the bowl over a pot of barely simmering water (double boiler) and whisk in the egg yolks until they are 
thickened and pale.',
    'Slowly drizzle the warm melted butter into the yolk mixture while whisking constantly. The sauce should 
emulsify and thicken; if it becomes too thick, add a teaspoon of water to loosen it.',
    'Remove the bowl from the heat and stir in the strained reduction. Taste and adjust seasoning with additional 
salt, pepper, or tarragon if desired.',
    'Keep the sauce warm by placing the bowl back over the double boiler or covering it with a warm towel. Do not 
let it boil or it will curdle.',
    'Serve immediately over steak, fish, or vegetables, garnished with a few fresh tarragon sprigs.'
]

Tips:
[
    'Use a double boiler to prevent the eggs from scrambling; the gentle, indirect heat is key.',
    'If the sauce separates, whisk in a teaspoon of cold water or a few drops of lemon juice to bring it back 
together.',
    'Fresh tarragon is essential for authentic flavor; dried tarragon can be used in a pinch but will be less 
potent.',
    'Keep the butter warm but not hot; if it’s too hot, the sauce will break.',
    'If you need to store the sauce, keep it in a thermos or a small saucepan over very low heat and stir 
occasionally to maintain consistency.'
]

## 11. Meta-Prompting

Meta-prompting involves creating prompts that help generate better prompts.

### Description
- Creates prompts for prompt generation
- Helps optimize prompt engineering
- Useful for complex or specialized tasks

### Example
```
Task: Create a prompt that will help generate creative story ideas.

Meta-prompt:
"Create a prompt that will:
1. Specify the genre and tone
2. Include key elements to consider
3. Provide structure for the story
4. Set clear constraints and requirements"
```

### Implementation


In [18]:

class PromptTemplate(BaseModel):
    title: str = Field(description="Title of the prompt template")
    purpose: str = Field(description="Purpose of the prompt")
    components: List[str] = Field(description="Components of the prompt")
    constraints: List[str] = Field(description="Constraints and requirements")

def meta_prompting_example():
    prompt = """
    Task: Create a prompt that will help generate creative story ideas.
    
    Create a prompt that will:
    1. Specify the genre and tone
    2. Include key elements to consider
    3. Provide structure for the story
    4. Set clear constraints and requirements
    """
    
    response = get_completion(
        prompt,
        temperature=0.7,
        response_model=PromptTemplate
    )
    rprint("Title:", response.title)
    rprint("Purpose:", response.purpose)
    rprint("Components:", response.components)
    rprint("Constraints:", response.constraints)

    second_response = get_completion(
        prompt=f""" Write a story using {response}""",
        temperature=0.7,
        response_model=None
    )
    rprint("Second Response:", second_response)

# Run the example
meta_prompting_example()

Title: Creative Story Idea Generator Prompt

Purpose: Assist writers in generating original and engaging story concepts by providing a structured framework that
covers genre, tone, essential elements, plot structure, and clear constraints.

Components:
[
    'Specify genre and tone',
    'Include key elements to consider',
    'Provide story structure outline',
    'Set clear constraints and requirements'
]

Constraints:
[
    'No use of copyrighted characters or settings',
    'Ideas must be original and not derivative',
    'Story concept limited to 200 words max',
    'Include at least three distinct main characters',
    'Specify a clear setting and time period',
    'Ensure a compelling twist or conflict'
]

Second Response: **Creative Story Idea Generator Prompt**  
*Assist writers in generating original and engaging story concepts by providing a structured framework that covers 
genre, tone, essential elements, plot structure, and clear constraints.*

In the rain‑slick streets of 1928 New York, a disillusioned librarian named **Elena** stumbles upon a dusty, 
unmarked manuscript in the archives. The book is a diary belonging to **Julian**, a time‑traveling historian who 
claims to have slipped into the past to prevent a forgotten tragedy. Elena, skeptical yet curious, enlists the help
of **Marla**, a charismatic street magician whose sleight of hand could reveal hidden truths.

**Genre & Tone**  
- Genre: Historical mystery with speculative tech.  
- Tone: Pensive yet hopeful, laced with urban grit.

**Key Elements to Consider**  
- A secret society guarding a temporal device.  
- Moral dilemma: altering history vs. preserving the present.  
- Urban noir aesthetics.

**Story Structure Outline**  
1. **Hook** – Elena finds the diary.  
2. **Rising Action** – Julian’s revelations, Marla’s tricks, a looming conspiracy.  
3. **Climax** – The trio confronts the society; Julian’s device hums to life.  
4. **Twist** – Marla is revealed to be Julian’s future self, orchestrating events to protect her own existence.  
5. **Resolution** – Elena chooses to keep the diary hidden, preserving the fragile timeline.

**Constraints & Requirements**  
- No copyrighted characters or settings.  
- Ideas must be original.  
- Story concept limited to 200 words.  
- At least three distinct main characters.  
- Clear setting: 1928 New York.  
- Compelling twist: Marla’s true identity as Julian’s future self.

## 12. Recursive Prompting

Recursive prompting uses the model's own outputs as inputs for subsequent prompts in an iterative process.

### Description
- Iterative refinement of responses
- Builds upon previous outputs
- Useful for complex, evolving tasks

### Example
```
Initial Prompt: "Generate a basic outline for a mystery novel"
[Model generates outline]

Follow-up Prompt: "Using this outline, expand the first chapter with more detail"
[Model expands the chapter]

Final Prompt: "Review the chapter and suggest improvements to the plot"
[Model provides feedback and suggestions]
```

### Implementation


In [19]:

class OutlineResponse(BaseModel):
    title: str = Field(description="Title of the story")
    chapters: List[str] = Field(description="Chapter outlines")

class ChapterResponse(BaseModel):
    chapter_number: int = Field(description="Chapter number")
    content: str = Field(description="Chapter content")
    word_count: int = Field(description="Word count")

class ReviewResponse(BaseModel):
    strengths: List[str] = Field(description="Strengths of the chapter")
    weaknesses: List[str] = Field(description="Areas for improvement")
    suggestions: List[str] = Field(description="Specific suggestions")

def recursive_prompting_example():
    # Step 1: Generate outline
    outline_prompt = "Generate a basic outline for a mystery novel"
    outline = get_completion(
        outline_prompt,
        temperature=0.7,
        response_model=OutlineResponse
    )
    rprint("Outline:", outline)
    
    # Step 2: Expand first chapter
    chapter_prompt = f"Using this outline, expand the first chapter with more detail:\n{outline}"
    chapter = get_completion(
        chapter_prompt,
        temperature=0.7,
        response_model=ChapterResponse
    )
    rprint("\nChapter:", chapter)
    
    # Step 3: Review and suggest improvements
    review_prompt = f"Review this chapter and suggest improvements to the plot:\n{chapter}"
    review = get_completion(
        review_prompt,
        temperature=0.7,
        response_model=ReviewResponse
    )
    rprint("\nReview:", review)

# Run the example
recursive_prompting_example()

Outline:
OutlineResponse(
    title='Shadows on Maple Street',
    chapters=[
        'Prologue: The Mysterious Disappearance',
        'Chapter 1: A New Detective Arrives',
        'Chapter 2: Clues in the Old Library',
        'Chapter 3: The Secret Passage Revealed',
        'Chapter 4: A Twist in the Family History',
        'Chapter 5: The Chase Through the Fog',
        'Chapter 6: The Final Confrontation',
        'Epilogue: The Truth Unveiled'
    ]
)

Chapter:
ChapterResponse(
    chapter_number=0,
    content='In the unassuming town of Maplewood, a chill wind cut through the rustling leaves of Maple Street, 
carrying with it a sense of unease that had begun to settle over the quiet community.\n\nOn a crisp October 
morning, the sun struggled to pierce the overcast sky, casting a muted glow over the row of Victorian houses that 
lined the street. In the most affluent corner of Maplewood, the Baines family had lived for generations, their 
legacy anchored by the old manor that stood proudly at the end of the block. The manor, with its grand columns and 
expansive gardens, was known for its opulence and the stories it held within its walls.\n\nThe disappearance of 
Mrs. Eleanor Baines, the matriarch of the family, had begun with a single, frantic call to the local police. Her 
son, Thomas, a man of sharp intellect and stoic demeanor, had found the front door locked from the inside, with no 
sign of forced entry. The only clue left behind was a single, crimson rose petal, half-buried in the threshold of 
the foyer, as if someone had left it purposely as a cryptic message.\n\nThe Baines household, known for its strict 
routine and meticulous organization, seemed to have been caught in a moment of silence. Thomas was visibly shaken; 
he had never seen his mother like this before. His wife, Claire, a former investigative journalist, was devastated.
Together, they were left to confront the mystery of a disappearance that seemed impossible: how could a woman who 
had spent her entire life in the safety of her estate, surrounded by her family, vanish without a trace?\n\nThe 
local police, led by Officer Ramirez, arrived at the scene to find the manor\'s doors locked, the windows securely 
closed, and no sign of struggle. The family’s teenage son, Ethan, had left the house hours earlier, claiming he had
caught a glimpse of his mother walking out of the back garden, but he had not recorded any details. The only 
tangible evidence was a single rose petal, a small but undeniably significant detail that the investigators could 
not ignore.\n\nThe lack of any obvious explanation spurred a frenzy of speculation among the townsfolk. Some 
whispered that the Baines family had long been involved in secret dealings that had finally caught up with them. 
Others suggested that a jealous lover or a disgruntled employee might have taken matters into their own hands. The 
local newspaper, the Maplewood Gazette, ran a front-page story that read, "Mrs. Baines Disappears: Who Will Uncover
the Truth?"\n\nAs the news spread, the town\'s sense of security wavered. The Baines\' estate, once a symbol of 
prestige and stability, now became a symbol of mystery and fear. The community, in the wake of the disappearance, 
found themselves on the brink of a crisis that would test their resilience and their willingness to confront the 
unknown.\n\nThe day after Mrs. Baines vanished, a shadowy figure was seen walking along Maple Street, flicking a 
long black coat as they passed by the Baines house. No one could confirm if the figure had any connection to the 
disappearance, but the sight left a lingering unease that would soon become a catalyst for the investigation that 
would unfold across the following chapters.\n\nThe prologue sets the stage for a narrative that combines family 
secrets, historical mysteries, and a relentless pursuit of truth. It is a story that will force the characters to 
confront their past and face the darkness that lurks beneath the normalcy of Maple Street.',
    word_count=586
)

Review:
ReviewResponse(
    strengths=[
        'Atmospheric opening that sets a moody, mysterious tone.',
        'Clear inciting incident – the sudden disappearance of Mrs. Baines.',
        'Strong, well‑defined characters (Thomas, Claire, Officer Ramirez).',
        'Use of a symbolic clue (the crimson rose petal) to hint at deeper secrets.',
        'Effective world‑building of Maplewood and the Baines manor.'
    ],
    weaknesses=[
        'Plot feels predictable and lacks a compelling twist or unique angle.',
        'The mystery is under‑developed; no concrete suspects or motives are presented.',
        'Pacing is slow and heavily exposition‑driven, leaving little room for suspense.',
        'The antagonist remains anonymous and unmotivated, reducing tension.',
        'Dialogue is sparse, limiting character depth and emotional engagement.'
    ],
    suggestions=[
        'Introduce early red‑herring clues that suggest multiple suspects (e.g., a hidden diary, a strange 
foot‑print).',
        'Show Thomas grappling with guilt and doubt through internal monologue or conversations with Claire, to 
deepen emotional stakes.',
        'Add sensory details of the manor’s interior (smell of old books, creaking floorboards) to enhance 
immersion.',
        'Use the rose petal as a recurring motif—perhaps it appears in other scenes, hinting at a pattern or a 
hidden symbol.',
        'Create a subtle antagonist presence: a shadowy figure in the garden, a cryptic note, or a mysterious 
neighbor who appears in key moments.',
        'Interweave family secrets (e.g., a long‑lost heir, a hidden will) to give the disappearance a stronger 
motive.',
        'Increase dialogue between the investigative journalist Claire and Officer Ramirez to reveal investigative 
strategies and build tension.',
        'Consider a flashback that reveals a past event in the Baines family that could tie into the disappearance,
adding depth and an emotional hook.'
    ]
)

## Best Practices

1. **Choose the Right Technique**
   - Use CoT for step-by-step reasoning
   - Use ToT when multiple approaches need to be considered
   - Use Few-shot for specific format requirements
   - Use ReAct when external tools are needed
   - Use Reflexion for self-improvement
   - Use Prompt Chaining for complex, multi-stage tasks

2. **Combine Techniques**
   - Different techniques can be combined for better results
   - Example: Use Few-shot with CoT for complex problems

3. **Iterate and Refine**
   - Test different prompt variations
   - Analyze model responses
   - Refine prompts based on performance

4. **Consider Context**
   - Adapt techniques to specific use cases
   - Account for model limitations
   - Consider input/output requirements

5. **Error Handling**
```python
def safe_completion(prompt: str, max_retries: int = 3) -> str:
    """Wrapper function with error handling"""
    for attempt in range(max_retries):
        try:
            return get_completion(prompt)
        except Exception as e:
            if attempt == max_retries - 1:
                raise e
            print(f"Attempt {attempt + 1} failed, retrying...")
```

6. **Temperature Control**
```python
def get_completion_with_temperature(prompt: str, task_type: str) -> str:
    """Helper function to set appropriate temperature based on task type"""
    temperature_map = {
        "creative": 0.9,
        "balanced": 0.7,
        "precise": 0.3
    }
    return get_completion(prompt, temperature=temperature_map.get(task_type, 0.7))
```

7. **Response Formatting**
```python
def format_response(response: str, format_type: str = "text") -> Any:
    """Format the response based on the desired output type"""
    if format_type == "json":
        try:
            return json.loads(response)
        except:
            return {"error": "Invalid JSON response"}
    return response
```

## Usage Tips

1. **Environment Setup**
   - Install required packages: `uv add openai instructor pydantic`
   - Set up your API key securely
   - Consider using environment variables for API keys

2. **Running Examples**
   - Each example can be run independently
   - Adjust temperature and other parameters as needed
   - Monitor API usage and costs

3. **Customization**
   - Modify prompts to suit your specific needs
   - Experiment with different parameters
   - Combine techniques for complex tasks

4. **Best Practices**
   - Always include error handling
   - Use appropriate temperature settings
   - Format responses consistently
   - Monitor API usage and costs
   - Use type hints and Pydantic models for better type safety
   - Leverage Instructor's structured output capabilities 